# DINOv3 full fine-tuning on Kaggle

Fresh initialization from the original `model.safetensors`: heads (5 epochs) → last 4 blocks (5 epochs) → fully unfrozen backbone (up to 100 epochs). The global checkpoint and stage-3 early stopping are selected by **val_unseen Recall@1**. Logs and final metrics include Accuracy, Recall@1/2/5/10, MRR and ranks.

Set `RESUME_CHECKPOINT` only for an explicit continuation from a previous notebook output. Kaggle inputs are read-only; all generated files stay under `/kaggle/working/VIscaNEr`.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['TQDM_MININTERVAL'] = '10'

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr')
RUN_NAME = 'dinov3_full_unseen_r1'
# First run: keep None. Continuation: point to the previous output's last.pt.
RESUME_CHECKPOINT = None

def find_unique(dataset_slug: str, filename: str) -> Path:
    matches = sorted(
        path for path in INPUT.rglob(filename)
        if path.is_file() and dataset_slug in path.parts
    )
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one {filename} inside {dataset_slug}; found: {matches}'
        )
    return matches[0]

full_trainer = find_unique('viscaner-dinov3-code', 'full_finetune.py')
code_root = full_trainer.parent
metadata = find_unique('viscaner-dinov3-data', 'crops_metadata.csv')
manifest = find_unique('viscaner-dinov3-data', 'bottle_images_manifest.csv')
weights = find_unique('viscaner-dinov3-vitb16-weights', 'model.safetensors')
assert metadata.parent == manifest.parent
assert (metadata.parent / 'crops').is_dir()
assert (metadata.parent / 'refs').is_dir()
assert (code_root / 'train_dinov3_retrieval.py').is_file()
assert (code_root / 'configs/dinov3_full_finetune.yaml').is_file()

shutil.copytree(
    code_root, PROJECT, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('dataset-metadata.json'),
)
checksum_path = PROJECT / 'code_sha256.json'
if checksum_path.is_file():
    checksums = json.loads(checksum_path.read_text(encoding='utf-8'))
    for filename, expected in checksums.items():
        candidate = PROJECT / filename
        actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
        assert actual == expected, f'Code integrity check failed: {filename}'
else:
    checksums = {}
    print('Warning: code_sha256.json is absent; file-presence checks still passed.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5.6,<6', 'safetensors>=0.5,<1',
    'scikit-learn>=1.5,<2', 'PyYAML>=6,<7',
], check=True)

import torch
import transformers
import yaml

assert torch.cuda.is_available(), 'Select a GPU accelerator before running this notebook'
print('CUDA:', torch.cuda.get_device_name(0), 'GPU count:', torch.cuda.device_count())
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
print('Training uses one GPU.')

config_path = PROJECT / 'configs/dinov3_full_finetune.yaml'
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg.update(
    project_root=str(PROJECT),
    weights_path=str(weights),
    crops_metadata_path=str(metadata),
    bottle_manifest_path=str(manifest),
    crops_root=str(metadata.parent / 'crops'),
    refs_root=str(metadata.parent / 'refs'),
    index_path=str(PROJECT / 'datasets/dinov3_retrieval/index.csv'),
    split_summary_path=str(PROJECT / 'datasets/dinov3_retrieval/split_summary.json'),
    models_dir=str(PROJECT / 'models' / RUN_NAME),
    runs_dir=str(PROJECT / 'runs/dinov3_retrieval'),
    run_name=RUN_NAME,
    device='cuda',
    amp=True,
)
assert cfg['stage1_epochs'] == 5
assert cfg['stage2_epochs'] == 5
assert cfg['stage3_epochs'] == 100
assert cfg['unfreeze_last_blocks'] == 4
assert cfg['early_stopping_split'] == 'val_unseen'
assert cfg['early_stopping_metric'] == 'recall_at_1'
assert weights.name == 'model.safetensors'
if RESUME_CHECKPOINT is None and any(Path(cfg['models_dir']).glob('*.pt')):
    raise RuntimeError(
        'Fresh run found existing checkpoints. Use a clean session or a new RUN_NAME.'
    )
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

(PROJECT / 'environment.json').write_text(json.dumps({
    'weights': str(weights),
    'code_input': str(code_root),
    'data_input': str(metadata.parent),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'initialization': (
        'original safetensors' if RESUME_CHECKPOINT is None else 'explicit continuation'
    ),
    'checkpoint_monitor': 'val_unseen.recall_at_1',
    'code_sha256': checksums,
}, indent=2), encoding='utf-8')
with (PROJECT / 'pip-freeze.txt').open('w', encoding='utf-8') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
print(json.dumps(cfg, indent=2))


## Verify all three stages on CUDA

This disposable smoke model performs a real forward/backward step with heads-only, partial-backbone and full-backbone trainability. Its weights are isolated and never reused by the full run.

In [ ]:
smoke_cfg = dict(cfg)
smoke_cfg.update(
    models_dir=str(PROJECT / 'smoke/models'),
    runs_dir=str(PROJECT / 'smoke/runs'),
    run_name='all_three_stages',
)
smoke_config = PROJECT / 'configs/smoke_full_finetune.yaml'
smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
subprocess.run([
    sys.executable, '-u', str(PROJECT / 'train_dinov3_retrieval.py'),
    'all', '--config', str(smoke_config), '--quick-smoke',
], cwd=PROJECT, check=True)
print('All three stages passed forward/backward checks. Smoke weights are isolated.')


## Start or continue full training

With `RESUME_CHECKPOINT = None`, this starts strictly from the supplied safetensors. Otherwise it validates and restores the exact full-training state from `last.pt`. Save the Kaggle notebook version to keep all outputs if the session budget requests continuation.

In [ ]:
command = [
    sys.executable, '-u', str(PROJECT / 'train_dinov3_retrieval.py'),
    'all', '--config', str(config_path),
]
if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT)
    assert resume_path.is_file(), f'Resume checkpoint not found: {resume_path}'
    command += ['--resume-checkpoint', str(resume_path)]
else:
    assert '--resume-checkpoint' not in command
print('Starting 5 head + 5 partial-backbone + up to 100 full-backbone epochs.', flush=True)
print('Checkpoint/early stopping: val_unseen Recall@1.', flush=True)
print('Stage 3: patience 12, minimum 15 epochs, warm-up 3 epochs.', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary_path = Path(cfg['runs_dir']) / RUN_NAME / 'run_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
if summary['continuation_required']:
    print(
        'Session budget reached. Save this output and continue from last.pt with '        'its sibling best.pt/best_full.pt.'
    )
else:
    print('Training completed. Best checkpoint:', summary['best_checkpoint'])
